03 · Scoring: is the agent right, and does the follow-up loop help?

Reads the run folders written by `02_agent_adk.ipynb` and scores them four ways:

| Part | Question | Answer key |
|---|---|---|
| 1. Labels | Do our labelers agree? (Cohen's κ) | the label sheets themselves |
| 2. Track A | Did the agent find the right clause? | CUAD lawyer highlights (`cuad_gold_spans.csv`) |
| 3. Track B | Is the verdict right? False alarms and missed risks per clause type | our gold labels |
| 4. Loop | When the follow-up loop changed a verdict, did it fix it or break it? What did it cost? | gold labels + traces |
| 5. Track C | Are the revision suggestions useful? | our 1–3 ratings |

Every table is also saved to `scores/` so the report and slides use the same numbers.

## 0. Setup

Put the run folders next to this notebook (unzip the `run.zip` files from notebook 02), named `{split}_{condition}`, e.g. `test_rag`, `test_single_pass`, `test_adaptive`.

In [ ]:
import sys, json, zipfile
from pathlib import Path
import pandas as pd
from sklearn.metrics import cohen_kappa_score

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import userdata, files
    if not Path("contract-review-agent").exists():
        token = userdata.get("GH_TOKEN")
        !git clone -q https://{token}@github.com/JJJJeny/contract-review-agent.git
    DATA_DIR = Path("contract-review-agent")
    for name in files.upload():                  # upload one or more run.zip files (optional)
        zipfile.ZipFile(name).extractall(".")
else:
    DATA_DIR = Path(".")

SPLIT = "test"                                   # "dev" while tuning, "test" for the final numbers
CONDITIONS = ["rag", "single_pass", "adaptive"]  # baseline -> loop off -> full agent
FAMILIES = ["payment", "termination", "liability_indemnity", "confidentiality", "renewal", "dispute_governing_law"]
OUT = Path("scores"); OUT.mkdir(exist_ok=True)


def find_run(cond):
    for root in (Path("."), Path("runs"), DATA_DIR / "runs"):
        if (root / f"{SPLIT}_{cond}" / "results.csv").exists():
            return root / f"{SPLIT}_{cond}"


RUNS = {c: find_run(c) for c in CONDITIONS}
RUNS = {c: d for c, d in RUNS.items() if d}
print("found runs:", {c: str(d) for c, d in RUNS.items()})
missing = set(CONDITIONS) - set(RUNS)
if missing:
    print("not found (skipped below):", sorted(missing))

pred = pd.concat([pd.read_csv(d / "results.csv").assign(condition=c) for c, d in RUNS.items()], ignore_index=True)


def save(df, name):
    df.to_csv(OUT / f"{name}.csv")
    return df

## 1. Labels and agreement (Cohen's κ)

Each label sheet has one row per contract × family. κ is agreement corrected for chance: (observed − chance) / (1 − chance).
Verdicts have no order, so plain κ. Severities do (n/a < low < medium < high), so weighted κ: being one level off costs less than being two levels off.

How our labels were made (say this in the report): both label sheets started as AI drafts. Each of us then reviewed the draft row by row, reading the quoted evidence, and wrote the final call in `human_verdict` (plus `why_changed` when we disagreed with the AI). Gold is what we agreed on after discussing every row where our reviewed verdicts differ.

So two numbers matter: **how many rows human review changed** (per reviewer), and **κ between the two reviewed sheets**.

In [ ]:
SHEETS = {                                            # one sheet per reviewer, plus gold when it exists
    "jenny": "labels_jenny_reviewed.xlsx",            # Claude draft + Jenny's human_verdict column
    "kayla": "labels_kayla_reviewed.xlsx",            # Kayla's AI draft + Kayla's human_verdict column
    "gold": "labels_gold.xlsx",
}
SEV = {"n/a": 0, "low": 1, "medium": 2, "high": 3}


def clean(col):
    return col.astype(str).str.strip().str.lower().replace({"nan": None, "": None})


def load_labels(path):
    """verdict = the human's call where they reviewed the row, else the AI draft's."""
    df = pd.read_excel(path).dropna(subset=["contract_id", "family"])
    df = df[df.family.isin(FAMILIES)].copy()
    df["ai_verdict"] = clean(df.verdict)
    df["reviewed"] = clean(df["human_verdict"]).notna() if "human_verdict" in df else False
    df["verdict"] = clean(df["human_verdict"]).fillna(df.ai_verdict) if "human_verdict" in df else df.ai_verdict
    sev = df["human_severity"] if "human_severity" in df else df.severity
    df["severity"] = clean(sev).fillna(clean(df.severity)).fillna("n/a")
    return df.set_index(["contract_id", "family"])


labels = {who: load_labels(DATA_DIR / f) for who, f in SHEETS.items() if (DATA_DIR / f).exists()}

review = pd.DataFrame({who: {"rows": len(df), "human_reviewed": int(df.reviewed.sum()),
                             "changed_by_human": int((df.reviewed & (df.verdict != df.ai_verdict)).sum())}
                       for who, df in labels.items() if who != "gold"}).T
display(save(review, "human_review"))
for who, df in labels.items():
    ch = df[df.reviewed & (df.verdict != df.ai_verdict)]
    if len(ch):
        print(f"\n{who} changed these AI verdicts:")
        display(ch[["ai_verdict", "verdict"] + (["why_changed"] if "why_changed" in ch else [])])


def kappa_table(a, b):
    cols = ["verdict", "severity", "reviewed"]
    m = labels[a][cols].join(labels[b][cols], lsuffix="_a", rsuffix="_b", how="inner")
    if (m.reviewed_a & m.reviewed_b).any():          # compare human calls only, once both have reviewed
        m = m[m.reviewed_a & m.reviewed_b]
    rows = {"all": m, **{f: m.xs(f, level="family") for f in FAMILIES}}
    return pd.DataFrame({
        k: {"rows": len(g),
            "verdict_agree": (g.verdict_a == g.verdict_b).mean(),
            "verdict_kappa": cohen_kappa_score(g.verdict_a, g.verdict_b),
            "severity_wkappa": cohen_kappa_score(g.severity_a.map(SEV), g.severity_b.map(SEV), weights="quadratic")}
        for k, g in rows.items()}).T.round(2), m


annotators = [w for w in labels if w != "gold"]
for i, a in enumerate(annotators):
    for b in annotators[i + 1:]:
        table, m = kappa_table(a, b)
        print(f"\n{a} vs {b}")
        display(save(table, f"kappa_{a}_vs_{b}"))
        display(pd.crosstab(m.verdict_a.rename(a), m.verdict_b.rename(b)))
        changed = m[m.verdict_a != m.verdict_b]
        print("rows where they disagree:", list(changed.index))

### Gold labels

`labels_gold.xlsx` is the answer key (columns `contract_id, family, verdict, severity`). Until it exists we fall back to Jenny's reviewed sheet, and every number below is marked provisional.

The rows where the labelers disagreed are kept as `hard`: we use them in part 4 to see whether the agent asks a human on exactly the rows humans found hard.

In [ ]:
GOLD_NAME = "gold" if "gold" in labels else "jenny"
gold = labels[GOLD_NAME][["verdict", "severity"]]
if GOLD_NAME != "gold":
    print("PROVISIONAL: no labels_gold.xlsx yet, scoring against", SHEETS[GOLD_NAME])

hard = set()
for i, a in enumerate(annotators):
    for b in annotators[i + 1:]:
        j = labels[a][["verdict", "reviewed"]].join(labels[b][["verdict", "reviewed"]], lsuffix="_a", rsuffix="_b", how="inner")
        if (j.reviewed_a & j.reviewed_b).any():
            j = j[j.reviewed_a & j.reviewed_b]
        hard |= set(j[j.verdict_a != j.verdict_b].index)
print(len(hard), "hard rows (labelers disagreed)")

## 2. Track A: clause detection against CUAD

CUAD only covers four of our families, and only part of each (see the README). For those, a lawyer highlighted the clause text. We count the agent's quote as a **hit** if it overlaps any CUAD highlight for that contract and family.

* **precision** = hits / rows where the agent quoted something
* **recall** = hits / rows where CUAD has a highlight
* **char F1** = how much of the highlighted text the quotes cover, character by character (a stricter, partial-credit measure)

A quote with no CUAD highlight is not always wrong: CUAD's categories are narrower than ours (e.g. termination for cause is not a CUAD category). So precision here is a lower bound.

In [ ]:
cuad = pd.read_csv(DATA_DIR / "cuad_gold_spans.csv")
cuad["span"] = list(zip(cuad.answer_start, cuad.answer_start + cuad.text.str.len()))
cuad_spans = cuad.groupby(["contract_id", "family"]).span.apply(list)
CUAD_FAMILIES = sorted(cuad.family.unique())


def chars(spans):
    return set().union(*(range(s, e) for s, e in spans)) if spans else set()


def track_a(df):
    rows = []
    for r in df[df.family.isin(CUAD_FAMILIES)].itertuples():
        got = [tuple(s) for s in json.loads(r.evidence_spans or "[]")]
        want = cuad_spans.get((r.contract_id, r.family), [])
        g, w = chars(got), chars(want)
        rows.append({"family": r.family, "quoted": bool(got), "has_cuad": bool(want),
                     "hit": bool(g & w), "char_tp": len(g & w), "char_pred": len(g), "char_gold": len(w)})
    t = pd.DataFrame(rows)

    def score(g):
        p = g.hit.sum() / max(g.quoted.sum(), 1)
        r = g.hit.sum() / max(g.has_cuad.sum(), 1)
        cp, cr = g.char_tp.sum() / max(g.char_pred.sum(), 1), g.char_tp.sum() / max(g.char_gold.sum(), 1)
        return pd.Series({"n": len(g), "precision": p, "recall": r, "f1": 2 * p * r / max(p + r, 1e-9),
                          "char_f1": 2 * cp * cr / max(cp + cr, 1e-9)})
    return pd.concat([t.groupby("family").apply(score), score(t).to_frame("all").T]).round(2)


track_a_tables = {c: track_a(pred[pred.condition == c]) for c in RUNS}
display(save(pd.concat(track_a_tables, names=["condition"]), "track_a_clause_detection"))

## 3. Track B: risk verdicts against our gold labels

From the small-business owner's side, the two mistakes that matter are:

* **false alarm**: the agent flags a clause (risky or missing) that gold says is acceptable. The owner wastes a negotiation point.
* **missed risk**: gold says risky or missing, and the agent says acceptable. The owner signs something bad. This is the worse mistake.

`unclear` is neither: the agent declined to decide. We report how often it does that, and whether it does it on rows gold also calls unclear.

In [ ]:
FLAG = {"risky", "missing"}


def rate(hits, total):
    return hits.sum() / total.sum() if total.sum() else float("nan")   # NaN, not 0, when nothing to count


def track_b(df):
    m = df.set_index(["contract_id", "family"])[["verdict"]].join(gold.verdict.rename("gold"), how="inner")

    def score(g):
        ok, flag = g.gold == "acceptable", g.gold.isin(FLAG)
        return pd.Series({
            "n": len(g),
            "accuracy": (g.verdict == g.gold).mean(),
            "false_alarm_rate": rate(ok & g.verdict.isin(FLAG), ok),
            "missed_risk_rate": rate(flag & (g.verdict == "acceptable"), flag),
            "missing_recall": rate((g.gold == "missing") & (g.verdict == "missing"), g.gold == "missing"),
            "said_unclear": (g.verdict == "unclear").mean(),
            "unclear_when_gold_unclear": rate((g.verdict == "unclear") & (g.gold == "unclear"), g.verdict == "unclear"),
        })
    by_fam = m.groupby(level="family").apply(score)
    return pd.concat([by_fam, score(m).to_frame("all").T]).round(2), m


track_b_tables, joined = {}, {}
for c in RUNS:
    track_b_tables[c], joined[c] = track_b(pred[pred.condition == c])
display(save(pd.concat(track_b_tables, names=["condition"]), "track_b_verdicts"))

The headline comparison, one row per condition. If the loop is worth its cost, `adaptive` should beat `single_pass` on missed risks without raising false alarms.

In [ ]:
headline = pd.DataFrame({c: t.loc["all"] for c, t in track_b_tables.items()}).T
for c in RUNS:
    s = pd.DataFrame(json.load(open(RUNS[c] / "summary.json"))["contracts"])
    headline.loc[c, "cost_usd"] = s.cost_usd.sum()
    headline.loc[c, "minutes"] = s.elapsed_s.sum() / 60
    headline.loc[c, "tool_calls"] = s.tool_calls.sum()
display(save(headline.round(3), "headline"))

ax = pd.DataFrame({c: t.drop("all")[["false_alarm_rate", "missed_risk_rate"]].stack() for c, t in track_b_tables.items()}) \
       .unstack().loc[:, (slice(None), "missed_risk_rate")].droplevel(1, axis=1) \
       .plot.bar(figsize=(8, 3.5), title="Missed risk rate by clause type (lower is better)", rot=20)
ax.figure.tight_layout(); ax.figure.savefig(OUT / "missed_risk_by_family.png", dpi=150)

## 4. Did the follow-up loop help?

In the adaptive run every row keeps the pre-pass verdict (`prepass_verdict`) next to the final one. For the rows the loop changed, compare both to gold:

* **fixed** wrong → right, **broke** right → wrong, **moved** wrong → a different wrong.

In [ ]:
if "adaptive" in RUNS:
    a = pred[pred.condition == "adaptive"].set_index(["contract_id", "family"]).join(gold.verdict.rename("gold"), how="inner")
    looped = a[a.path != "prepass"]
    changed = looped[looped.verdict_changed.astype(str).str.lower() == "true"]
    before_ok, after_ok = changed.prepass_verdict == changed.gold, changed.verdict == changed.gold
    flips = pd.Series({
        "rows": len(a), "entered_loop": len(looped), "verdict_changed": len(changed),
        "fixed (wrong->right)": int((~before_ok & after_ok).sum()),
        "broke (right->wrong)": int((before_ok & ~after_ok).sum()),
        "moved (wrong->wrong)": int((~before_ok & ~after_ok).sum()),
        "prepass_accuracy": (a.prepass_verdict == a.gold).mean().round(2),
        "final_accuracy": (a.verdict == a.gold).mean().round(2),
    })
    display(save(flips.to_frame("count"), "loop_flips"))
    display(changed[["prepass_verdict", "verdict", "gold", "triggers_fired", "steps"]])

### Trajectories: what the loop did

Actions per family, how often the bounds stopped it, and which rows went to the human queue. Rows the labelers disagreed on (`hard`) are where we would want the agent to ask a human.

In [ ]:
if "adaptive" in RUNS:
    tr = pd.read_json(RUNS["adaptive"] / "trace.jsonl", lines=True)
    loop_steps = tr[tr.phase.eq("loop")] if "phase" in tr else tr
    display(save(loop_steps.action.value_counts().to_frame("calls"), "action_mix"))
    print("steps per looped family:", looped.steps.describe()[["mean", "50%", "max"]].round(2).to_dict())

    q = pd.DataFrame(json.load(open(RUNS["adaptive"] / "human_queue.json")))
    if len(q):
        q["kind"] = q.question.str.extract(r"Agent stopped \(([^:)]+)")[0].fillna("agent asked")
        q["hard_row"] = [(r.contract_id, r.family) in hard for r in q.itertuples()]
        display(save(q.groupby("kind").agg(n=("family", "size"), on_hard_rows=("hard_row", "sum")), "human_queue"))
    asked = {(r.contract_id, r.family) for r in q.itertuples()} if len(q) else set()
    hard_in_run = hard & set(a.index)
    print(f"hard rows sent to a human: {len(hard_in_run & asked)} / {len(hard_in_run)};",
          f"easy rows sent to a human: {len(asked - hard)} / {len(set(a.index) - hard)}")

## 5. Track C: are the revision suggestions useful?

Each of us rates every revision-plan item from 1 to 3 on three questions, without looking at the other's sheet:

* **grounded**: does it follow from the quoted clause?
* **specific**: does it say what to change (a number, a notice period, a cap)?
* **actionable**: could the owner send it to the vendor as is?

Run the next cell once to write the rating sheet, fill it in (one copy each, e.g. `ratings_jenny.csv`, `ratings_kayla.csv`), then run the cell after.

In [ ]:
if "adaptive" in RUNS:
    plans = json.load(open(RUNS["adaptive"] / "revision_plans.json"))
    sheet = pd.DataFrame([{"contract_id": cid, "id": it["id"], "family": it["family"], "severity": it.get("severity"),
                           "evidence": it.get("evidence_text", "")[:300], "ask": it.get("ask", ""),
                           "fallback": it.get("fallback", ""), "grounded": "", "specific": "", "actionable": ""}
                          for cid, items in plans.items() for it in items])
    sheet.to_csv(OUT / "ratings_template.csv", index=False)
    print(len(sheet), "plan items to rate ->", OUT / "ratings_template.csv")

In [ ]:
CRITERIA = ["grounded", "specific", "actionable"]
rated = {p.stem.split("_", 1)[1]: pd.read_csv(p).set_index(["contract_id", "id"])
         for p in sorted(Path(".").glob("ratings_*.csv")) if p.stem != "ratings_template"}
if rated:
    display(save(pd.DataFrame({who: r[CRITERIA].mean() for who, r in rated.items()}).round(2), "track_c_means"))
    if len(rated) == 2:
        (a_name, a_r), (b_name, b_r) = rated.items()
        both = a_r[CRITERIA].join(b_r[CRITERIA], lsuffix="_a", rsuffix="_b", how="inner").dropna()
        display(save(pd.Series({k: cohen_kappa_score(both[k + "_a"].astype(int), both[k + "_b"].astype(int),
                                                      weights="quadratic") for k in CRITERIA}).round(2)
                     .to_frame("weighted_kappa"), "track_c_agreement"))
else:
    print("no ratings_*.csv yet")

## 6. Save

Everything in `scores/` goes into the submission ZIP and the report.

In [ ]:
print(sorted(p.name for p in OUT.iterdir()))
if IN_COLAB:
    !zip -qr scores.zip scores
    files.download("scores.zip")